# NOTEBOOK PARA LEER DATOS

In [ ]:
import sys
import torch
sys.path.append("/home/ninjaraiz/anaconda3/repos/pyLowOrder/")

from FotR import SAM
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'The gun: {device}')

### Datos DLR

In [ ]:
dataset_dir = '/home/m.jaraiz/repos/CETACEO_UPM/use_cases/aerodynamics/KAN/data_DLR/'
train_file = dataset_dir + "train.h5"
test_file = dataset_dir + "test.h5"
val_file = dataset_dir + "val.h5"

keys = ['features/cp', 'features/alpha', 'features/mach', 'mesh/xyz']
path_files = dict({'train': train_file, 'test': test_file, 'val': val_file})

def leer_datos(path_file, keys):
    reader = SAM.HDF5reader(path_file)
    dic = {key: reader.load_to_tensor(key, False) for key in keys}
    #dic[keys[0]] = dic[keys[0]].transpose(0, 1)
    tensor_cp = dic[keys[0]].transpose(0, 1)
    tensor_flcc = torch.cat((dic[keys[1]], dic[keys[2]]), dim=1)
    tensor_ptos = dic[keys[3]]
    tensor_ptos = tensor_ptos[:, [0, 2]]
    
    return dict({'ptos': tensor_ptos, 'flcc': tensor_flcc, 'cp': tensor_cp})

train_dict = leer_datos(train_file, keys=keys)
test_dict = leer_datos(test_file, keys=keys)
val_dict = leer_datos(val_file,keys=keys)

complete_dict = {}
for key in train_dict.keys():
    if key == 'cp':
        complete_dict[key] = torch.cat((train_dict[key], test_dict[key], val_dict[key]), dim=1)
    elif key == 'ptos':
        complete_dict[key] = train_dict[key]
    else:
        complete_dict[key] = torch.cat((train_dict[key], test_dict[key], val_dict[key]), dim=0)
print(f'complete_dict: {complete_dict.keys()}')
for key in complete_dict.keys():
    print(f'{key}: {complete_dict[key].shape}')

tensor_ptos = complete_dict['ptos'] # save_tensor_to_geo(tensor_flcc, "perfil_importado.geo", alpha=0.5)
tensor_flcc = complete_dict['flcc']

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 1, figsize=(10, 10))
ax[0].scatter(
    tensor_ptos[:,0],
    tensor_ptos[:,1],
    c=torch.linspace(1,tensor_ptos.shape[0], tensor_ptos.shape[0]),
    cmap='coolwarm',
    s=10
)
ax[0].set_title('ptos')

ax[1].scatter(tensor_flcc[:, 0].cpu(), tensor_flcc[:, 1].cpu(), s=10)
ax[1].set_title('flcc')
ax[1].set_xlabel('alpha')
ax[1].set_ylabel('mach')
plt.show()

### Datos F22

In [ ]:
dataset_dir = "./airfoil_files/"
import pandas as pd

df = pd.read_csv(dataset_dir + "eta_0_0.csv", sep=',', decimal='.')

# Suponiendo que las columnas son: '# ID', 'X', 'Y', 'Z'
# Ignorar la columna de ID
datos = df[['X', 'Y', 'Z']].values  # Extraer como arreglo NumPy

# Convertir a tensor de torch
tensor_ptos = torch.tensor(datos, dtype=torch.float32)
tensor_ptos = tensor_ptos[:,[0,2]]

In [ ]:
print(tensor_ptos[:,0].max() - tensor_ptos[:,0].min())

## GENERACIÓN DE ARCHIVO .GEO PARA GMSH

In [ ]:
import numpy as np
from scipy.spatial import Delaunay

def alpha_shape(points, alpha):
    def add_edge(edges, edge_points, coords, i, j):
        if (i, j) in edges or (j, i) in edges:
            edges.discard((j, i))
            edges.discard((i, j))
        else:
            edges.add((i, j))
            edge_points.add((tuple(coords[i]), tuple(coords[j])))

    tri = Delaunay(points)
    edges = set()
    edge_points = set()

    for ia, ib, ic in tri.simplices:
        pa, pb, pc = points[ia], points[ib], points[ic]
        A = np.linalg.norm(pb - pc)
        B = np.linalg.norm(pa - pc)
        C = np.linalg.norm(pa - pb)
        s = (A + B + C) / 2.0
        area = max(np.sqrt(s * (s - A) * (s - B) * (s - C)), 1e-10)
        circum_r = A * B * C / (4.0 * area)

        if circum_r < alpha:
            add_edge(edges, edge_points, points, ia, ib)
            add_edge(edges, edge_points, points, ib, ic)
            add_edge(edges, edge_points, points, ic, ia)

    if not edge_points:
        raise ValueError("No se encontraron aristas con el valor de alpha dado.")

    edge_points = list(edge_points)
    start = np.array(edge_points[0][0])
    ordered_points = [start]
    for _ in range(len(edge_points)):
        for i, (s, e) in enumerate(edge_points):
            s = np.array(s)
            e = np.array(e)
            if np.allclose(ordered_points[-1], s):
                ordered_points.append(e)
                edge_points.pop(i)
                break
            elif np.allclose(ordered_points[-1], e):
                ordered_points.append(s)
                edge_points.pop(i)
                break
    return np.array(ordered_points)

def sort_tensor(
        tensor: torch.Tensor,
        alpha: float = 1.0
):
    if tensor.shape[1] != 2:
        raise ValueError("El tensor debe tener dos columnas (x, y).")
    points = tensor.numpy()
    sorted_points = alpha_shape(points, alpha)
    return torch.tensor(sorted_points, dtype=torch.float32)

def sort_tensor_by_angle(
        tensor: torch.Tensor,
):
    center = torch.mean(tensor, axis=0)
    angles = torch.arctan2(tensor[:, 1] - center[1], tensor[:, 0] - center[0])
    return tensor[torch.argsort(angles)]

def save_tensor_airfoil_to_geo(
        tensor: torch.Tensor,
        filename: str,
        str_lc: str = 'lc_a',
        start_enumeration:int = 1001,
        print:bool = True
    ):
    if tensor.shape[1] != 2:
        raise ValueError("El tensor debe tener dos columnas (x, y).")
    x_min, x_max = torch.min(tensor[:, 0]), torch.max(tensor[:, 0])
    c = x_max - x_min
    x_quarter = x_min + c / 4
    quarter_indices = torch.where(torch.isclose(tensor[:, 0], x_quarter, atol=1e-3))[0]
    if len(quarter_indices) >= 2:
        y_values = tensor[quarter_indices, 1]
        t = torch.max(y_values) - torch.min(y_values)
    else:
        t = 0.0
    tensor = tensor.unique_consecutive(dim=0)
    #Centrando el perfil
    tensor = tensor - torch.mean(tensor, dim=0)
    
    n_points = tensor.shape[0]
    if print:
        with open(filename, "w") as f:
            f.write("// Archivo generado automáticamente con los puntos importados\n")
            f.write("SetFactory(\"OpenCASCADE\");\n\n")
            f.write(f"c = {c};\n")
            f.write(f"n_points = {n_points};\n")
            f.write(f"t = {t};\n")
            # f.write(f"{str_lc} = 1e-3;\n")
            
            for i, (x, y) in enumerate(tensor, start=start_enumeration):
                f.write(f"Point({i}) = {{{x}, 0.0, {y}, {str_lc}}};\n")

    return tensor

def save_tensor_to_geo(
        tensor: torch.Tensor,
        filename: str,
        str_lc: str = 'lc_a',
        start_enumeration:int = 1001,
        print:bool = True
    ):
    if tensor.shape[1] != 2:
        raise ValueError("El tensor debe tener dos columnas (x, y).")

    tensor = tensor.unique_consecutive(dim=0)
    
    n_points = tensor.shape[0]
    if print:
        with open(filename, "w") as f:
            f.write("// Archivo generado automáticamente\n")
            # f.write("SetFactory(\"OpenCASCADE\");\n\n")
            # f.write(f"c = {c};\n")
            f.write(f"n_points_{start_enumeration} = {n_points};\n")
            # f.write(f"t = {t};\n")
            # f.write(f"{str_lc} = 1e-3;\n")
            
            for i, (x, y) in enumerate(tensor, start=start_enumeration):
                f.write(f"Point({i}) = {{{x}, 0.0, {y}, {str_lc}}};\n")

    return tensor

def close_trailing_edge_tensor(tensor: torch.Tensor) -> torch.Tensor:
    n = tensor.shape[0]

    extrados = tensor[tensor[:,1] > 0, :]
    intrados = tensor[tensor[:,1] < 0, :]

    # Buscar el par de puntos con menor x en cada cara
    p1, p2 = extrados[extrados[:, 0].argsort()[:2]]
    p3, p4 = intrados[intrados[:, 0].argsort()[:2]]

    v1 = p2 - p1  # Dirección de la recta del extradós
    v2 = p4 - p3  # Dirección de la recta del intradós

    A = torch.stack([v1, -v2], dim=1)  # Sistema A @ [t, s] = p3 - p1
    b = p3 - p1

    try:
        ts = torch.linalg.solve(A, b)
        intersection = p1 + ts[0] * v1
    except RuntimeError:
        intersection = 0.5 * (p2 + p4)

    return torch.cat([tensor, intersection.unsqueeze(0)], dim=0)

In [ ]:
points1 = save_tensor_airfoil_to_geo(sort_tensor_by_angle(close_trailing_edge_tensor(tensor_ptos.double())), "perfil_sorted_importado.geo", str_lc = 'lc_a', start_enumeration=1001, print=False)

import matplotlib.pyplot as plt
plt.figure(figsize=(20, 5))
plt.scatter(
    points1[:,0],
    points1[:,1],
    c=torch.linspace(0, points1.shape[0], points1.shape[0]).cpu(),
    cmap='coolwarm',
    s=10
)
plt.show()


In [ ]:
def generate_wake_refinement_points(
    x_start: float,
    x_end: float,
    n_x: int,
    n_y: int,
    wake_width: float,
    growth_type: str = "parabolic",
    device="cpu",
    ):
    """
    Genera nube de puntos para refinamiento de wake.

    Parameters
    ----------
    x_start : float
        Inicio de la wake.

    x_end : float
        Final de la wake.

    n_x : int
        Nº de puntos longitudinales.

    n_y : int
        Nº de puntos transversales.

    wake_width : float
        Anchura máxima de la wake.

    growth_type : str
        "cone" o "parabolic"

    Returns
    -------
    pts : torch.Tensor
        Tensor (N,2)
    """

    xs = torch.linspace(
        x_start,
        x_end,
        n_x,
        device=device
    )

    all_pts = []

    L = x_end - x_start

    for x in xs:

        xi = (x - x_start) / L

        # crecimiento transversal
        if growth_type == "cone":

            local_half_width = wake_width * xi

        elif growth_type == "parabolic":

            local_half_width = wake_width * torch.sqrt(xi)

        else:
            raise ValueError(
                f"Unknown growth_type: {growth_type}"
            )

        ys = torch.linspace(
            -local_half_width,
            local_half_width,
            n_y,
            device=device
        )

        xx = torch.full_like(ys, x)

        pts_i = torch.stack(
            [xx, ys],
            dim=1
        )

        all_pts.append(pts_i)

    pts = torch.cat(all_pts, dim=0)

    return pts


points2 = generate_wake_refinement_points(
    x_start = -0.4,
    x_end   = 1,
    n_x     = 20,
    n_y     = 5,
    wake_width = 1.5,
    growth_type = "parabolic",
)

In [ ]:
# points2 tiene que hacer una parabola con centro en (0, 0), parámetro "a" y eje x como semieje mayor
points2 = torch.tensor([[0.6 * (x**2) - 0.15, x] for x in torch.linspace(-1.5, 1.5, 50)], dtype=torch.float64)
# points2 = torch.tensor([[0.6 * (x**2) - 0.35, x] for x in torch.linspace(-1.5, 1.5, 50)], dtype=torch.float64)

In [ ]:
# points2 = torch.cat([points2, torch.tensor([[0.5, 0], [-0.5, 0]], dtype=torch.float32)], dim=0)
save_tensor_to_geo(points2, "./includes/parabola.geo", str_lc = 'lc_ff', start_enumeration=2001, print=True)
points2